In [2]:
!python -m pip install --upgrade pip --default-timeout=300

In [3]:
!python -m pip install mlflow --timeout 1200 --retries 10 --no-cache-dir

In [5]:
!pip install imblearn


   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   -------------------------- ------------- 2/3 [imblearn]
   ---------------------------------------- 3/3 [imblearn]



In [5]:
!pip install awscli boto3 python-dotenv 

In [14]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [36]:
# !aws configure set aws_access_key_id os.getenv("aws_access_key_id")
# !aws configure set aws_secret_access_key os.getenv("aws_secret_access_key")
# !aws configure set region "ap-south-1"

In [37]:
# import mlflow

# mlflow.set_tracking_uri("http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/")

In [38]:
# mlflow.set_experiment("Exp 4 - Handling Imbalance Data")

In [18]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn

In [19]:
df = pd.read_csv("cleaned_df.csv")
df.columns

Index(['Unnamed: 0', 'clean_comment', 'category'], dtype='object')

In [20]:
df.drop(columns=['Unnamed: 0'], inplace=True)

In [21]:
df

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1
...,...,...
36788,jesus,0
36789,kya bhai pure saal chutiya banaya modi aur jab...,1
36790,downvote karna tha par upvote hogaya,0
36791,haha nice,1


In [22]:
df['category'].value_counts()

category
 1    15771
 0    12772
-1     8250
Name: count, dtype: int64

In [25]:
df.isna().sum()

clean_comment    131
category           0
dtype: int64

In [26]:
df.dropna(inplace=True)
df.isna().sum()

clean_comment    0
category         0
dtype: int64

In [33]:
def imbalance_data_handling_experiment(imbalance_method):
    ngram_range = (1,3)
    max_features = 500

    X = df['clean_comment']
    Y = df['category']

    x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=.2, random_state=42, stratify=Y)

    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    x_train_vec = vectorizer.fit_transform(x_train)
    x_test_vec = vectorizer.transform(x_test)


    if imbalance_method == "class_weights":
        class_weight = "balanced"
    else:
        class_weight = None

        if imbalance_method == "oversampling":
            smote = SMOTE(random_state=42)
            x_train_vec, y_train = smote.fit_resample(x_train_vec, y_train)
        elif imbalance_method == "adasyn":
            adasyn = ADASYN(random_state=42)
            x_train_vec, y_train = adasyn.fit_resample(x_train_vec, y_train)
        elif imbalance_method == "undersampling":
            rus = RandomUnderSampler(random_state=42)
            x_train_vec, y_train = rus.fit_resample(x_train_vec, y_train)
        elif imbalance_method == "smote_enn":
            smote_enn = SMOTEENN(random_state=42)
            x_train_vec, y_train = smote_enn.fit_resample(x_train_vec, y_train)

    with mlflow.start_run() as run:
        mlflow.log_param("Vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("max_features", max_features)

        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("imbalance_method", imbalance_method)
        
        model = RandomForestClassifier(n_estimators=n_estimators ,max_depth=max_depth, random_state=42, class_weight=class_weight)
        model.fit(x_train_vec, y_train)
        y_pred = model.predict(x_test_vec)

        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)


        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8,6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"confusion matrix: RF model TF-IDF(1,3)_{max_features}_{imbalance_method}")
        plt.savefig(f"Confusion_matrix_{imbalance_method}.png")
        mlflow.log_artifact(f"Confusion_matrix_{imbalance_method}.png")
        plt.close()

        mlflow.sklearn.log_model(
            model,
            name= f"RF_model_TFIDF(1,3)_{max_features}_{imbalance_method}",
            skops_trusted_types=["sklearn.tree._tree.Tree"]
        )

    print("10*=")
    print(f"Accuracy - {imbalance_method}: {accuracy}")
    print("10*=")

        

In [34]:
imbalance_methods = ["class_weights", "oversampling", "adasyn", "undersampling", "smote_enn"]

for imbalance_method in imbalance_methods:
    imbalance_data_handling_experiment(imbalance_method)

🏃 View run trusting-wren-29 at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/4/runs/17e760f399c44a619673bca97b4dcb31
🧪 View experiment at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/4
10*=
Accuracy - class_weights: 0.6658939042683758
10*=
🏃 View run dazzling-mouse-769 at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/4/runs/64d12f83431749fab6009d8c7af56b90
🧪 View experiment at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/4
10*=
Accuracy - oversampling: 0.6589390426837584
10*=
🏃 View run youthful-squid-875 at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/4/runs/3e33002576ce43d3a101ef73adefd63d
🧪 View experiment at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.com:5000/#/experiments/4
10*=
Accuracy - adasyn: 0.6709395881630983
10*=
🏃 View run debonair-panda-798 at: http://ec2-43-204-97-155.ap-south-1.compute.amazonaws.c